## Agenda

1. Recall: `FORMAT`, `PROC SORT`, and the DATA step
2. Conditional processing with `IF`/`THEN`/`ELSE`
3. Running more than one statement with `DO` and `END`
4. Plotting frequencies with `PROC FREQ`
5. Plotting with `PROC SGPLOT`

## Recall

| | |
|---|---|
| **`FORMAT`** | tells SAS how to display numeric, character, date, or datetime values when we print or output our data |
| **`PROC SORT`** | sorts our data, and is a way to remove duplicates from our data |
| **DATA step** | creates new SAS tables, filters rows, computes new columns, and uses SAS functions |

## Conditional processing

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<u><b>IF</b> <i>expression</i> <b>THEN</b> <i>statement</i>;</u><br>
&nbsp;&nbsp;&nbsp;&nbsp;<span class="fragment">&lt;<b>ELSE IF</b> <i>expression</i> <b>THEN</b> <i>statement</i>;&gt;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&lt;<b>ELSE</b> <i>statement</i>;&gt;</span><br>
<b>RUN;</b></div>
```

- `IF` checks each row. If the expression is true, SAS runs the statement after `THEN`.
- `ELSE IF` and `ELSE` are optional. SAS stops at the first one that is true.

## `IF`/`THEN`: example

Let's label each car as `Okay` or `Too High` based on its price.

In [ ]:
/* if-then in a data step */
data cars_new;
    set sashelp.cars;
    if MSRP < 30000 then Cost = "Okay";
    if MSRP >= 30000 then Cost = "Too High";
    keep Make Model Type MSRP Cost;
run;

proc print data=cars_new (obs=10);
run;

::: {.note style="display:inline-block"}
Look at `Cost`. It says **Too** instead of **Too High**. SAS sets the length of a new column from the **first** value it sees. Here that is `"Okay"`, so `Cost` only has room for 4 characters.
:::

## `IF`/`THEN`: setting the length

We fix this with a `LENGTH` statement **before** the column is used. `$` means character, and `10` is the number of characters.

In [ ]:
/* if-then in a data step */
data cars_new;
    set sashelp.cars;
    length Cost $ 10;
    if MSRP < 30000 then Cost = "Okay";
    else Cost = "Too High";
    keep Make Model Type MSRP Cost;
run;

proc print data=cars_new (obs=10);
run;

We also changed the second `IF` to `ELSE`. Every car that is not under 30,000 must be 30,000 or more, so we don't need to check again.

## More than one statement: `DO` and `END`

```{=html}
<div class="syntax-box"><b>DATA</b> <i>output-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>SET</b> <i>input-table</i>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>IF</b> <i>expression</i> <b>THEN DO</b>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&lt;<i>multiple statements</i>&gt;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>END</b>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&lt;<b>ELSE IF</b> <i>expression</i> <b>THEN DO</b>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&lt;<i>multiple statements</i>&gt;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>END</b>;&gt;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&lt;<b>ELSE DO</b>;<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&lt;<i>multiple statements</i>&gt;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>END</b>;&gt;<br>
<b>RUN;</b></div>
```

`THEN` only runs **one** statement. To run more than one, wrap them in `DO;` ... `END;`.

## `DO` and `END`: example

We can also write rows to **different tables**. List all the tables on the `DATA` statement, then use `OUTPUT` to say where each row goes.

In [ ]:
/* if-then extension */
data under40 over40;
    set sashelp.cars;
    keep Make Model MSRP Cost;
    if MSRP < 20000 then do;
        Cost = 1;
        output under40;
    end;
    else if MSRP < 40000 then do;
        Cost = 2;
        output under40;
    end;
    else do;
        Cost = 3;
        output over40;
    end;
run;

Check the log to see how many rows went to each table. You can also open **WORK.UNDER40** and **WORK.OVER40** in the Libraries pane.

::: {.note style="display:inline-block"}
Once we use `OUTPUT`, SAS only writes a row when it reaches an `OUTPUT` statement.
:::

## Exercise

Use the Overwatch stats table `ma505.ow_stats` (not the heroes table).

1. Create a new column `HighestVal` with the highest value of `Damage`, `Heals`, and `Mitigation`. Create another column `Category` that says which one it was.

    | Damage | Heals | Mitigation | HighestVal | Category |
    |---|---|---|---|---|
    | 10432 | 2345 | 985 | 10432 | DAMAGE |

2. Change the code so that rows where `Category` is DAMAGE go to a new table called `damage`, HEALS to a table called `heals`, and MITIGATION to a table called `mitigation`.
    a. How many observations are in each new table?

Save the full table with `Category` as `ma505.ow_cat`. We will use it in the next exercise.

<div class="solution-start"></div>

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

data ma505.ow_cat damage heals mitigation;
    set ma505.ow_stats;
    length Category $ 10;
    HighestVal = max(Damage, Heals, Mitigation);
    if HighestVal = Damage then do;
        Category = "DAMAGE";
        output damage;
    end;
    else if HighestVal = Heals then do;
        Category = "HEALS";
        output heals;
    end;
    else do;
        Category = "MITIGATION";
        output mitigation;
    end;
    output ma505.ow_cat;    /* every row also goes to the full table */
run;

/* how many rows in each category? */
proc freq data=ma505.ow_cat;
    tables Category;
run;

## Plotting data with PROC steps

So far our procedures have given us **tables**. Many of them can also make **plots**.

We will start with `PROC FREQ`, then move to `PROC SGPLOT`.

## `PROC FREQ` with a plot

```{=html}
<div class="syntax-box"><b>PROC FREQ DATA</b>=<i>input-table</i> &lt;<i>proc-options</i>&gt;;<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>TABLES</b> <i>col-name(s)</i> &lt;/ <i>options</i>&gt;;<br>
<b>RUN;</b></div>
```

Recall: what does this code do? Let's extend it by adding a plot of the frequencies.

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="libref-circle"><span class="kw">ods graphics on</span></span>;<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">ODS Graphics needs to be on to make plots. It stays on until the end of the session.</span></span>

<span class="kw">proc freq</span> data=sashelp.cars;
    <span class="kw">tables</span> Origin /
        plots=freqplot(<span class="libref-circle">orient=horizontal</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">creates a horizontal bar plot</span></span>
                       <span class="libref-circle">scale=percent</span>);<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">shows percents instead of counts</span></span>
<span class="kw">run</span>;</pre></div></div>
```

In [ ]:
/* creating a frequency plot */
ods graphics on;

proc freq data=sashelp.cars;
    tables Origin /
        plots=freqplot(orient=horizontal scale=percent);
run;

## Two-way frequency table

Use `*` between two columns to count every **combination** of their values.

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">proc freq</span> data=sashelp.cars;
    <span class="kw">tables</span> <span class="libref-circle">Origin*Type</span> /<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">creates a two-way frequency table</span></span>
        plots=freqplot(<span class="libref-circle">orient=vertical</span>);<span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">vertical bar chart. We did not set <span class="mono">scale=</span>, so the plot shows counts.</span></span>
<span class="kw">run</span>;</pre></div></div>
```

We don't need `ods graphics on` again, since it is still on from the last code.

In [ ]:
/* creating a vertical bar freq plot */
proc freq data=sashelp.cars;
    tables Origin*Type /
        plots=freqplot(orient=vertical);
run;

## Stacked bar chart

`twoway=stacked` puts the bars for each `Origin` on top of each other.

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="libref-circle"><span class="kw">title1</span> 'Origin of Cars by Type';</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text"><span class="mono">TITLE</span> and <span class="mono">FOOTNOTE</span> add text to the whole report, not just the table or plot.</span></span>
<span class="kw">title2</span> 'Sedan, SUV, Truck';
<span class="kw">footnote</span> 'Other types not included: Hybrid, Sports, Wagon';
<span class="kw">proc freq</span> data=sashelp.cars;
    <span class="kw">tables</span> Origin*Type /
        plots=freqplot(<span class="libref-circle">twoway=stacked</span> orient=horizontal);
    <span class="kw">where</span> Type in ("Sedan", "SUV", "Truck");
<span class="kw">run</span>;
<span class="libref-circle"><span class="kw">title</span>; <span class="kw">footnote</span>;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">Titles stay on until we clear them. This clears them.</span></span></pre></div></div>
```

In [ ]:
/* creating a stacked bar */
title1 'Origin of Cars by Type';
title2 'Sedan, SUV, Truck';
footnote 'Other types not included: Hybrid, Sports, Wagon';

proc freq data=sashelp.cars;
    tables Origin*Type /
        plots=freqplot(twoway=stacked orient=horizontal);
    where Type in ("Sedan", "SUV", "Truck");
run;

title;
footnote;

## Exercise

1. Using `ma505.ow_cat` from the last exercise, create a **clustered** vertical bar chart of each player's `Category`.
    a. Set it up like the stacked bar chart, but change `stacked` to `cluster`.
2. What do the 4 values in each cell of the frequency table mean?

<div class="solution-start"></div>

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

proc freq data=ma505.ow_cat;
    tables Player*Category /
        plots=freqplot(orient=vertical twoway=cluster);
run;

Each cell of a two-way table has 4 values, in this order:

- **Frequency**: the number of games for that player and category.
- **Percent**: that count out of **all** games.
- **Row Pct**: out of that **player's** games (each row adds to 100).
- **Col Pct**: out of that **category's** games (each column adds to 100).

## `PROC SGPLOT`

`PROC SGPLOT` makes many kinds of statistical graphics: scatter plots, bar charts, line plots, histograms, box plots, and more. It is part of SAS ODS Graphics.

Compared to the plots in `PROC FREQ`, it gives us:

- **more flexibility** over how the plot looks
- **layering**: more than one plot on the same axes

## `PROC SGPLOT` syntax

```{=html}
<div class="anno"><div class="anno-row"><pre class="anno-code"><span class="kw">proc sgplot</span> data=<i>input-table</i> &lt;<i>options</i>&gt;;
    <span class="libref-circle"><i>plot statement(s)</i> / &lt;<i>options</i>&gt;;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">what plot to add (one plot or layered plots), and options for that plot</span></span>
    <span class="libref-circle">&lt;<i>appearance statements</i>&gt;;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">legend, appearance, axes, symbols</span></span>
    <span class="libref-circle">&lt;<i>additional filtering/formatting/etc.</i>&gt;;</span><span class="anno-inline fragment"><span class="arrow-left"></span><span class="anno-inline-text">any extra data processing, like <span class="mono">WHERE</span> or <span class="mono">FORMAT</span></span></span>
<span class="kw">run</span>;</pre></div></div>
```

## Plot statements

```{=html}
<div class="attr-grid" style="grid-template-columns: repeat(2, minmax(0, 1fr));">
  <div class="attr-col"><h4 style="color:#1565c0">Basic</h4><div class="attr-card a">BAND, BUBBLE, BLOCK, DROPLINE, ELLIPSEPARM, FRINGE, <mark>HBARPARM</mark>, HEATMAP, HEATMAPPARM, HIGHLOW, LINEPARM, NEEDLE, POLYGON, REFLINE, SCATTER, SERIES, SPLINE, STEP, TEXT, VBARPARM, VECTOR</div></div>
  <div class="attr-col"><h4 style="color:#c62828">Distribution</h4><div class="attr-card b">DENSITY, HBOX, HISTOGRAM, VBOX</div></div>
  <div class="attr-col"><h4 style="color:#e65100">Fit and Confidence</h4><div class="attr-card b">ELLIPSE, LOESS, PBSPLINE, REG</div></div>
  <div class="attr-col"><h4 style="color:#2e7d32">Categorization</h4><div class="attr-card a">DOT, <mark>HBAR</mark>, HBARBASIC, HLINE, VBAR, VBARBASIC, VLINE, WATERFALL</div></div>
</div>
```

::: {.fragment}
Only certain groups can be **layered** together: Basic with Fit and Confidence, Distribution with Distribution, and Categorization with Categorization.
:::

::: {.fragment}
- `HBARPARM` assumes the data is **already summarized** (one row per bar).
- `HBAR` assumes the data is **not summarized** (many rows per bar). SAS does the counting or adding for us.
:::

## `PROC SGPLOT`: a simple bar chart

With only a category column, `HBAR` counts the rows for each value.

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

proc sgplot data=ma505.ow_stats;
    hbar Map;
run;

## Adding a response variable

`response=` tells SAS what to measure for each bar. By default it **adds up** (sums) the response.

In [ ]:
proc sgplot data=ma505.ow_stats;
    hbar Map / response=Damage;
run;

## Choosing the statistic

`stat=` changes how the response is summarized, like `stat=mean` for the average.

In [ ]:
proc sgplot data=ma505.ow_stats;
    hbar Map / response=Damage stat=mean;
run;

## Exercise

1. Using `ma505.ow_stats`, make a bar chart of the **sum of `Damage` by `Date`** two ways: with `VBAR` and with `VBARPARM`. Compare the outputs.
    a. Are the results what you expected? Why or why not?

<div class="solution-start"></div>

In [ ]:
libname ma505 "~";    /* skip if your autoexec does this */

/* VBAR: SAS adds up Damage for each date */
proc sgplot data=ma505.ow_stats;
    vbar Date / response=Damage stat=sum;
run;

/* VBARPARM: expects one row per date, so it does not add anything up */
proc sgplot data=ma505.ow_stats;
    vbarparm category=Date response=Damage;
run;

`VBAR` gives the sum of `Damage` for each date. `VBARPARM` does not. It is a **Basic** plot that expects the data to be summarized already, with one row per bar. Our table has several games on each date, so the bars are drawn on top of each other and do not show the total. To use `VBARPARM` here, we would need to summarize the data first.